# Model Understanding: Who Is Likely To Default?

This notebook helps you answer:

- Which customer patterns are associated with default?
- What does the default population look like vs non-default?
- Which features separate risky vs safer customers most clearly?

It uses training statements (`train_data`) + labels (`train_labels`) to create customer-level behavioral summaries and visualizations.


## 1) Setup


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 120)


## 2) Config

Update `DATA_DIR` if your data lives elsewhere.


In [ ]:
# Try local repo data folder first, then Colab path.
CANDIDATE_DIRS = [
    Path('../data/raw'),
    Path('data/raw'),
    Path('/content/drive/MyDrive/amex_data_parquet'),
]

DATA_DIR = next((p for p in CANDIDATE_DIRS if p.exists()), CANDIDATE_DIRS[0])

TRAIN_DATA_FILE = DATA_DIR / 'train_data.parquet'
TRAIN_DATA_CSV = DATA_DIR / 'train_data.csv'
TRAIN_LABELS_FILE = DATA_DIR / 'train_labels.csv'

# Optional downsampling for faster exploratory runs.
SAMPLE_N_CUSTOMERS = 80_000  # set to None for full dataset
RANDOM_STATE = 42

print('Using DATA_DIR:', DATA_DIR.resolve())
print('train_data.parquet exists:', TRAIN_DATA_FILE.exists())
print('train_data.csv exists:', TRAIN_DATA_CSV.exists())
print('train_labels.csv exists:', TRAIN_LABELS_FILE.exists())


## 3) Load Data


In [ ]:
def load_train_data(parquet_path: Path, csv_path: Path) -> pd.DataFrame:
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(
        f'Could not find train data. Checked: {parquet_path} and {csv_path}'
    )

if not TRAIN_LABELS_FILE.exists():
    raise FileNotFoundError(f'Missing labels file: {TRAIN_LABELS_FILE}')

train = load_train_data(TRAIN_DATA_FILE, TRAIN_DATA_CSV)
labels = pd.read_csv(TRAIN_LABELS_FILE)

train['customer_ID'] = train['customer_ID'].astype('string')
labels['customer_ID'] = labels['customer_ID'].astype('string')
train['S_2'] = pd.to_datetime(train['S_2'])

print('train shape:', train.shape)
print('labels shape:', labels.shape)
print('statement date range:', train['S_2'].min(), '->', train['S_2'].max())
print('unique customers:', train['customer_ID'].nunique())


## 4) Build Customer-Level Understanding Table

We combine:

- `last statement snapshot` per customer
- behavior summaries (`n_statements`, `history_days`, missingness)
- target (`default`)


In [ ]:
# Optional sampling (for faster iteration)
if SAMPLE_N_CUSTOMERS is not None:
    all_customers = train['customer_ID'].drop_duplicates()
    sampled_customers = all_customers.sample(
        n=min(SAMPLE_N_CUSTOMERS, len(all_customers)), random_state=RANDOM_STATE
    )
    train_use = train[train['customer_ID'].isin(sampled_customers)].copy()
    labels_use = labels[labels['customer_ID'].isin(sampled_customers)].copy()
    print(f'Using sampled customers: {train_use.customer_ID.nunique():,}')
else:
    train_use = train.copy()
    labels_use = labels.copy()
    print(f'Using full customers: {train_use.customer_ID.nunique():,}')

train_use = train_use.sort_values(['customer_ID', 'S_2'])

# Last statement snapshot for each customer
last_snapshot = train_use.groupby('customer_ID', as_index=False).tail(1).copy()

# Behavioral features
g = train_use.groupby('customer_ID')

behavior = pd.DataFrame({
    'customer_ID': g.size().index,
    'n_statements': g.size().values,
    'history_days': (g['S_2'].max() - g['S_2'].min()).dt.days.values,
    'avg_row_missing_rate': g.apply(lambda x: x.isna().mean(axis=1).mean()).values,
})

customer_df = (
    last_snapshot
    .merge(behavior, on='customer_ID', how='left')
    .merge(labels_use, on='customer_ID', how='inner')
)

customer_df['target'] = customer_df['target'].astype(int)
print('customer_df shape:', customer_df.shape)
customer_df[['target', 'n_statements', 'history_days', 'avg_row_missing_rate']].head()


## 5) Target Balance


In [ ]:
default_rate = customer_df['target'].mean()
print(f'Default rate: {default_rate:.3%}')

ax = sns.countplot(data=customer_df, x='target', palette='Set2')
ax.set_title('Class Balance (0 = Non-default, 1 = Default)')
ax.set_xlabel('target')
ax.set_ylabel('customer count')
plt.show()


## 6) Behavioral Pattern Comparison

These plots compare default vs non-default groups on engineered behavior signals.


In [ ]:
plot_cols = ['n_statements', 'history_days', 'avg_row_missing_rate']

fig, axes = plt.subplots(1, len(plot_cols), figsize=(5 * len(plot_cols), 4))
if len(plot_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, plot_cols):
    sns.boxplot(data=customer_df, x='target', y=col, ax=ax, palette='Set2')
    ax.set_title(f'{col} by target')

plt.tight_layout()
plt.show()

summary = customer_df.groupby('target')[plot_cols].median().rename(index={0: 'non_default', 1: 'default'})
summary


## 7) Find Strong Single-Feature Signals

For each numeric feature, we compute a univariate ROC-AUC against `target`.

- AUC near `0.5`: weak separation
- AUC far from `0.5` (toward `0` or `1`): stronger separation


In [ ]:
numeric_cols = customer_df.select_dtypes(include=[np.number]).columns.tolist()
ignore_cols = {'target'}
feature_cols = [c for c in numeric_cols if c not in ignore_cols]

auc_rows = []
for c in feature_cols:
    x = customer_df[c]
    valid = x.notna()
    if valid.sum() < 200:
        continue
    if x[valid].nunique() < 2:
        continue
    try:
        auc = roc_auc_score(customer_df.loc[valid, 'target'], x[valid])
        strength = abs(auc - 0.5)
        auc_rows.append((c, auc, strength, valid.mean()))
    except Exception:
        pass

auc_df = pd.DataFrame(auc_rows, columns=['feature', 'auc', 'strength', 'coverage']).sort_values('strength', ascending=False)
auc_df.head(20)


In [ ]:
top_n = 15
top_features = auc_df.head(top_n).copy()

plt.figure(figsize=(8, max(4, top_n * 0.35)))
sns.barplot(data=top_features, y='feature', x='strength', palette='viridis')
plt.title('Top Univariate Separation Strength (|AUC - 0.5|)')
plt.xlabel('separation strength')
plt.ylabel('feature')
plt.tight_layout()
plt.show()

top_features[['feature', 'auc', 'coverage']]


## 8) Risk Curves for Top Features

This shows how default rate changes from low to high values (quantile bins).


In [ ]:
def plot_default_rate_by_quantile(df: pd.DataFrame, col: str, q: int = 10):
    tmp = df[[col, 'target']].dropna().copy()
    if tmp[col].nunique() < q:
        print(f'Skipping {col}: not enough unique values for {q} bins')
        return

    tmp['bin'] = pd.qcut(tmp[col], q=q, labels=False, duplicates='drop')
    rate = tmp.groupby('bin')['target'].mean().reset_index()

    plt.figure(figsize=(6, 3.5))
    sns.lineplot(data=rate, x='bin', y='target', marker='o')
    plt.title(f'Default Rate by {col} Quantile')
    plt.xlabel('quantile bin (low -> high)')
    plt.ylabel('default rate')
    plt.ylim(0, min(1, rate['target'].max() * 1.2))
    plt.tight_layout()
    plt.show()

for f in top_features['feature'].head(5):
    print(f'Feature: {f}')
    plot_default_rate_by_quantile(customer_df, f, q=10)


## 9) (Optional) Quick Model for Feature Importance

This trains a lightweight model for directional understanding, not leaderboard performance.


In [ ]:
try:
    import lightgbm as lgb

    model_features = [
        c for c in customer_df.columns
        if c not in {'customer_ID', 'S_2', 'target'}
    ]

    X = customer_df[model_features].copy()
    y = customer_df['target'].copy()

    # Keep only numeric columns for a quick, robust baseline.
    X = X.select_dtypes(include=[np.number])

    # Simple missing-value handling.
    X = X.fillna(X.median(numeric_only=True))

    X_train, X_valid, y_train, y_valid = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )

    clf = lgb.LGBMClassifier(
        n_estimators=350,
        learning_rate=0.05,
        num_leaves=64,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
    )
    clf.fit(X_train, y_train)

    valid_pred = clf.predict_proba(X_valid)[:, 1]
    auc = roc_auc_score(y_valid, valid_pred)
    print(f'Quick model validation AUC: {auc:.4f}')

    fi = (
        pd.DataFrame({'feature': X.columns, 'importance': clf.feature_importances_})
        .sort_values('importance', ascending=False)
        .head(20)
    )

    plt.figure(figsize=(8, 7))
    sns.barplot(data=fi, y='feature', x='importance', palette='mako')
    plt.title('Quick Model: Top Feature Importances')
    plt.tight_layout()
    plt.show()

    fi

except ImportError:
    print('lightgbm is not installed in this environment. Skipping quick model section.')


## 10) Interpretation Notes

Use this section to write your own findings after running the notebook.

Suggested prompts:

1. Which 3-5 features show the strongest risk separation?
2. Do defaults show consistent differences in statement history length or missingness?
3. Are top risk curves monotonic (risk increases smoothly with feature value) or nonlinear?
4. Which insights are stable when you change `SAMPLE_N_CUSTOMERS`?
